In [1]:
import uuid
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import random
from datetime import datetime, timedelta
from faker import Faker
from pyspark.sql.functions import col, when, date_sub, current_timestamp, datediff, max, min
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DateType, TimestampType, DecimalType

In [2]:
def generate_users(fake: Faker, count: int):
    rows = []
    for i in range(1, count + 1):
        rows.append({
            "user_id": i,
            "email": fake.unique.email(),
            "name": fake.name(),
            "age": fake.random_int(min=10, max=100),
            "gender": random.choice(["M", "F"]),
            "job": fake.job(),
            "address": fake.address(),
            "signup": (datetime.now() - timedelta(days=random.randint(0, 365))).date(),
            "created_at": datetime.now()
        })
    return rows


def generate_orders(user_ids: list[int], count: int):
    statuses = ["CREATED", "PAID", "SHIPPED", "DELIVERED", "CANCELED"]
    rows = []
    for _ in range(count):
        status = random.choice(statuses)
        created_at = datetime.now() - timedelta(days=random.randint(0, 60))
        rows.append({
            "order_no": f"ORD-{uuid.uuid4().hex[:20].upper()}",
            "user_id": int(random.choice(user_ids)),
            "status": status,
            "total_amount": Decimal(random.randint(1_000, 500_000)) / Decimal(100),
            "created_at": created_at,
            "updated_at": created_at
        })
    return rows


user_schema = StructType([
    StructField("user_id", LongType(), False),
    StructField("email", StringType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("job", StringType(), True),
    StructField("address", StringType(), True),
    StructField("signup", DateType(), True),
    StructField("created_at", TimestampType(), True)
])

order_schema = StructType([
    StructField("order_no", StringType(), False),
    StructField("user_id", LongType(), False),
    StructField("status", StringType(), False),
    StructField("total_amount", DecimalType(14, 2), False),
    StructField("created_at", TimestampType(), False),
    StructField("updated_at", TimestampType(), False),
])

In [3]:
spark = SparkSession.builder.appName("S3 Spark") \
    .master("spark://spark-master.mmix.io:7077") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "1000000") \
    .config("spark.hadoop.fs.s3a.vectored.read.min.seek.size", "4K") \
    .config("spark.hadoop.fs.s3a.vectored.read.max.merged.size", "1M") \
    .config("spark.hadoop.fs.s3a.vectored.active.ranged.reads", "4") \
    .config("spark.hadoop.fs.s3a.experimental.input.fadvise", "random") \
    .getOrCreate()

26/06/14 20:13:15 WARN Utils: Your hostname, genius.local resolves to a loopback address: 127.0.0.1; using 192.168.45.182 instead (on interface en0)
26/06/14 20:13:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/14 20:13:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/14 20:13:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/14 20:13:16 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/06/14 20:13:16 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/06/14 20:13:16 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [4]:
users_ = generate_users(Faker("ko_KR"), 10)
users = spark.createDataFrame(users_, schema=user_schema)
orders = spark.createDataFrame(generate_orders([user["user_id"] for user in users_], 10), schema=order_schema)

In [5]:
users.show(1), orders.show(1)
users.select("user_id", "name", "gender").show(1), orders.select("order_no", "status", "total_amount").show(1)

+-------+--------------------+------+---+------+-----------------+--------------------------------+----------+--------------------+
|user_id|               email|  name|age|gender|              job|                         address|    signup|          created_at|
+-------+--------------------+------+---+------+-----------------+--------------------------------+----------+--------------------+
|      1|jeongsun91@exampl...|민정남| 94|     F|낙농업관련 종사원|광주광역시 강동구 도산대14거리 9|2025-12-08|2026-06-14 20:13:...|
+-------+--------------------+------+---+------+-----------------+--------------------------------+----------+--------------------+
only showing top 1 row

+--------------------+-------+-------+------------+--------------------+--------------------+
|            order_no|user_id| status|total_amount|          created_at|          updated_at|
+--------------------+-------+-------+------------+--------------------+--------------------+
|ORD-A37BE5EF85594...|      4|CREATED|       29.37|2026-06

(None, None)

In [6]:
users.count(), orders.count()
orders.select("user_id").distinct().count()

7

26/06/14 20:13:28 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [7]:
users.filter(col("age") >= 30).count()
orders.filter(col("status") == "PAID").count()
orders.filter((col("status") == "DELIVERED") & (col("total_amount") >= 100_000)).count()

0

In [8]:
orders.orderBy(col("total_amount").desc()).show(1)
users.orderBy("age").show(1)

+--------------------+-------+---------+------------+--------------------+--------------------+
|            order_no|user_id|   status|total_amount|          created_at|          updated_at|
+--------------------+-------+---------+------------+--------------------+--------------------+
|ORD-57F66BD974FB4...|      3|DELIVERED|     4983.79|2026-04-20 20:13:...|2026-04-20 20:13:...|
+--------------------+-------+---------+------------+--------------------+--------------------+
only showing top 1 row

+-------+--------------------+------+---+------+------------+---------------------------------+----------+--------------------+
|user_id|               email|  name|age|gender|         job|                          address|    signup|          created_at|
+-------+--------------------+------+---+------+------------+---------------------------------+----------+--------------------+
|      3|seojungim@example...|강우진| 18|     F|배우 및 모델|부산광역시 영등포구 압구정가 6...|2026-01-20|2026-06-14 20:13:...|
+----

In [9]:
orders.withColumn(
    "amount_level",
    when(col("total_amount") >= 300_000, "HIGH")
    .when(col("total_amount") >= 100_000, "MID")
    .otherwise("LOW")) \
    .select("order_no", "total_amount", "amount_level") \
    .show(1)

+--------------------+------------+------------+
|            order_no|total_amount|amount_level|
+--------------------+------------+------------+
|ORD-A37BE5EF85594...|       29.37|         LOW|
+--------------------+------------+------------+
only showing top 1 row



### 최근 7일 주문만 분석

In [10]:
recent_orders = orders.filter(col("created_at") >= date_sub(current_timestamp(), 7))
recent_orders.groupBy("status").count().show(1)

+------+-----+
|status|count|
+------+-----+
+------+-----+



### 간단한 조인

In [11]:
u = users.alias("u")
o = orders.alias("o")
users_orders = u.join(o, on="user_id", how="inner").select(col("user_id"), col("name"), col("signup"), col("order_no"), col("o.created_at"))
users_orders.show(1)

+-------+------+----------+--------------------+--------------------+
|user_id|  name|    signup|            order_no|          created_at|
+-------+------+----------+--------------------+--------------------+
|      2|박선영|2025-12-09|ORD-9E9873F140F74...|2026-04-18 20:13:...|
+-------+------+----------+--------------------+--------------------+
only showing top 1 row



### 사용자 가입 후 첫 주문까지 걸린 시간

In [12]:
first_order = users.groupBy("user_id", "name", "signup").agg(F.min("created_at").alias("first_order_at"))
first_order.withColumn("days_to_first_order", datediff("first_order_at", "signup")).select("user_id", "name", "days_to_first_order").show(10)

+-------+------+-------------------+
|user_id|  name|days_to_first_order|
+-------+------+-------------------+
|      1|민정남|                188|
|      2|박선영|                187|
|      6|고경수|                176|
|      5|박경수|                355|
|      3|강우진|                145|
|      4|정영길|                 42|
|      7|이숙자|                214|
|     10|박정호|                 45|
|      9|김영미|                337|
|      8|박지현|                169|
+-------+------+-------------------+



In [12]:
spark.stop()